# 📓 Notebook 03 — Pipelines em Tempo de Treinamento com Streaming

> **Pergunta operacional:** como treinar quando o dado não cabe na RAM ou chega em fluxo contínuo?
>
> **Origem canônica:** Disciplina 02, Aula 04 (loop de treinamento) + Disciplina 04, Aula 02 (bibliotecas).

## Objetivo

Mostrar três regimes de treino e quando usar cada um:

| Regime | Quem é o iterador | Para que serve |
|--------|-------------------|----------------|
| **Batch tradicional** (`fit`) | sklearn carrega tudo | dataset cabe na RAM, foco em qualidade |
| **Mini-batch / partial_fit** (`SGDClassifier`) | nós (gerador Python) | dataset não cabe na RAM, mas é finito |
| **Online / streaming** (`river`) | fluxo contínuo de eventos | dado nunca para de chegar, modelo aprende incrementalmente |

Tudo aqui usa o **mesmo Gold do notebook 02** — a diferença está no laço de treino, não no pré-processamento conceitual.

## 0. Setup

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Iterator

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from river import compose, linear_model, metrics, preprocessing
    RIVER_AVAILABLE = True
except ImportError:
    RIVER_AVAILABLE = False

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

RAW_PATH = Path("../dataset/processed/telco_churn.csv")
df = pd.read_csv(RAW_PATH)
df["churn_bin"] = (df["churn"] == "Yes").astype(int)

NUM_COLS = ["tenure", "monthly_charges", "total_charges"]
CAT_COLS = ["contract", "internet_service", "payment_method"]
TARGET = "churn_bin"

print(f"Linhas: {len(df):,}")
print("river disponível?", RIVER_AVAILABLE)

Linhas: 8,000
river disponível? True


## 1. Linha de base — batch tradicional

Treinamos um `LogisticRegression` com todo o dataset na RAM. É o ponto de comparação.

In [2]:
X_full = pd.get_dummies(df[NUM_COLS + CAT_COLS], columns=CAT_COLS, drop_first=False)
y_full = df[TARGET].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=RANDOM_SEED
)

scaler_batch = StandardScaler().fit(X_train[NUM_COLS])
X_train_s = X_train.copy()
X_test_s = X_test.copy()
X_train_s[NUM_COLS] = scaler_batch.transform(X_train[NUM_COLS])
X_test_s[NUM_COLS] = scaler_batch.transform(X_test[NUM_COLS])

batch_clf = LogisticRegression(max_iter=500, random_state=RANDOM_SEED).fit(X_train_s, y_train)
batch_auc = roc_auc_score(y_test, batch_clf.predict_proba(X_test_s)[:, 1])
print(f"AUC batch tradicional : {batch_auc:.3f}")

AUC batch tradicional : 0.956


## 2. Mini-batch — `partial_fit` com gerador

Quando o dataset não cabe na RAM, lemos em **chunks**. O `SGDClassifier(loss="log_loss")` aprende incrementalmente via `partial_fit`. O gerador abaixo seria substituído por `pd.read_csv(..., chunksize=...)` ou um `IterableDataset` do PyTorch.

In [3]:
def chunked_loader(df: pd.DataFrame, chunk_size: int = 512) -> Iterator[tuple[pd.DataFrame, np.ndarray]]:
    """Simula um leitor de arquivo em chunks. Mantém ordem temporal estável."""
    for start in range(0, len(df), chunk_size):
        chunk = df.iloc[start : start + chunk_size]
        yield chunk, chunk[TARGET].to_numpy()

sgd = SGDClassifier(loss="log_loss", random_state=RANDOM_SEED, learning_rate="optimal")
scaler_inc = StandardScaler()
first_fit_done = False

train_df, holdout_df = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=RANDOM_SEED)
feature_template = pd.get_dummies(df[NUM_COLS + CAT_COLS], columns=CAT_COLS, drop_first=False).columns

for chunk, y_chunk in chunked_loader(train_df, chunk_size=512):
    feats = pd.get_dummies(chunk[NUM_COLS + CAT_COLS], columns=CAT_COLS, drop_first=False)
    feats = feats.reindex(columns=feature_template, fill_value=0)
    if not first_fit_done:
        scaler_inc.partial_fit(feats[NUM_COLS])
    feats[NUM_COLS] = scaler_inc.transform(feats[NUM_COLS])
    sgd.partial_fit(feats, y_chunk, classes=np.array([0, 1]))
    first_fit_done = True

feats_h = pd.get_dummies(holdout_df[NUM_COLS + CAT_COLS], columns=CAT_COLS, drop_first=False)
feats_h = feats_h.reindex(columns=feature_template, fill_value=0)
feats_h[NUM_COLS] = scaler_inc.transform(feats_h[NUM_COLS])
minibatch_auc = roc_auc_score(holdout_df[TARGET], sgd.decision_function(feats_h))
print(f"AUC mini-batch (SGD) : {minibatch_auc:.3f}")

AUC mini-batch (SGD) : 0.950


### 🛑 Breakpoint — discussão

1. Por que precisamos do `reindex` com `fill_value=0` em cada chunk?
2. Onde o `scaler_inc` ainda tem risco de leakage? Como mitigar em produção?
3. Em quais situações `SGDClassifier` perde para uma rede neural com PyTorch DataLoader?

## 3. Online learning — `river`

Quando o dado nunca para, o `river` aprende **um evento por vez** e mantém uma métrica corrente (`progressive validation`).

In [4]:
if not RIVER_AVAILABLE:
    print("ℹ️  river não instalado — pulando esta seção. Instale com: pip install river")
else:
    model = compose.Pipeline(
        preprocessing.StandardScaler() | linear_model.LogisticRegression()
    )
    metric = metrics.ROCAUC()

    stream_df = train_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    history: list[float] = []
    for _, row in stream_df.iterrows():
        x = {
            "tenure": float(row["tenure"]),
            "monthly_charges": float(row["monthly_charges"]),
            "total_charges": float(row["total_charges"]),
        }
        y = int(row[TARGET])
        proba = model.predict_proba_one(x).get(True, 0.5)
        metric.update(bool(y), proba)
        model.learn_one(x, bool(y))
        history.append(metric.get())

    print(f"AUC online final : {metric.get():.3f}")
    print("AUC online após 1.000 eventos:", round(history[999], 3) if len(history) > 999 else "(stream curto)")

AUC online final : 0.827
AUC online após 1.000 eventos: 0.802


## 4. Comparação dos três regimes

O ranking depende do problema, mas a tabela serve para sinalizar o trade-off que cada equipe vai discutir.

In [5]:
comparison = pd.DataFrame(
    {
        "regime": ["batch (fit)", "mini-batch (partial_fit)", "online (river)"],
        "AUC holdout": [
            round(batch_auc, 3),
            round(minibatch_auc, 3),
            round(metric.get(), 3) if RIVER_AVAILABLE else float("nan"),
        ],
        "requer dataset em RAM?": ["sim", "não", "não"],
        "aprende em produção?": ["não", "opcional", "sim"],
        "caminho típico no TC": [
            "baseline + comparação",
            "feature store em lotes",
            "detector + re-treino contínuo",
        ],
    }
)
comparison

,regime,AUC holdout,requer dataset em RAM?,aprende em produção?,caminho típico no TC
0,batch (fit),0.956,sim,não,baseline + comparação
1,mini-batch (partial_fit),0.950,não,opcional,feature store em lotes
2,online (river),0.827,não,sim,detector + re-treino contínuo


## 5. Tracking com MLflow (opcional)

Mesmo no encontro 01 já vale registrar parâmetros e métricas — sem isso, não há rastreabilidade. Se `mlflow` não estiver disponível, o bloco apenas imprime os mesmos campos.

In [6]:
experiment = {
    "experiment": "encontro-01-streaming",
    "runs": [
        {"name": "batch", "params": {"max_iter": 500}, "metrics": {"auc": float(batch_auc)}},
        {"name": "minibatch", "params": {"chunk_size": 512}, "metrics": {"auc": float(minibatch_auc)}},
    ],
}
if RIVER_AVAILABLE:
    experiment["runs"].append(
        {"name": "online", "params": {"events": len(stream_df)}, "metrics": {"auc": float(metric.get())}}
    )

try:
    import mlflow

    mlflow.set_experiment(experiment["experiment"])
    for run in experiment["runs"]:
        with mlflow.start_run(run_name=run["name"]):
            mlflow.log_params(run["params"])
            mlflow.log_metrics(run["metrics"])
    print("✅ Runs registradas via MLflow.")
except Exception as exc:
    print("ℹ️  MLflow indisponível, imprimindo payload:", exc)
    print(experiment)

C:\Users\ricar\Github\mlet\fases\fase-01-produtizacao-de-modelos\eventos\grupos-de-estudo\encontro-01\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/05/25 17:16:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/05/25 17:16:04 INFO mlflow.store.db.utils: Updating database tables


2026/05/25 17:16:05 INFO mlflow.tracking.fluent: Experiment with name 'encontro-01-streaming' does not exist. Creating a new experiment.


✅ Runs registradas via MLflow.


## 🧠 Exercícios

**Iniciante.** Mude o `chunk_size` para 64 e depois 4.096. O que muda na AUC? E no tempo?

**Intermediário.** Troque `SGDClassifier` por `MLPClassifier` (rede pequena). Garanta que o `partial_fit` continua sendo possível e meça a degradação ao reduzir o número de épocas.

**Avançado.** Construa um pequeno `IterableDataset` do PyTorch que carregue o CSV em chunks e alimente um `nn.Sequential([Linear, ReLU, Linear, Sigmoid])`. Compare com o `SGD` aqui — qual é mais robusto a mudança de distribuição entre chunks?

## ➡️ Próximo notebook

Modelo treinado, próximo passo: **provar que ele serve**. O [`04_profiling_pos_treinamento.ipynb`](04_profiling_pos_treinamento.ipynb) ataca os splits treino/teste/homologação com profiling e análise por fatia.